In [1]:
import os
from dotenv import load_dotenv
load_dotenv()


True

##summerization Middleware

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

In [5]:
agent = create_agent(
    model = "gpt-4.1-nano",
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("messages",10),
            keep=("messages",4),
        ),
        #next middleware
    ]
    
    
)

In [6]:
### Run with thread

config = {"configurable": {"thread_id":"test-1"}}

In [7]:
questions =[
    "What is 4x4?",
    "What is 4x5?",
    "What is 4x6?",
    "What is 4x7?",
    "What is 4x8?",
    "What is 4x9?",
    "What is 4x10?",
    "What is 4x411",
    "What is 4x1?",
    "What is 4x2?",
    "What is 4x3?",
    "What is 4x445?",
    "What is 4x45?",
]

In [9]:
for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Len: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 4x4?', additional_kwargs={}, response_metadata={}, id='75f7cbb8-b59d-40ed-8eda-da3fd5077cdc'), AIMessage(content='4×4 equals 16.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 14, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_185fd20ed4', 'id': 'chatcmpl-DFQVeXb0XjQ4ix83ww60IKZOly9fe', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cb55b-6ba1-7852-80e4-0b4a6a5a68ef-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 7, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': 

#### from Token size

In [11]:
from langchain_core.tools import tool


@tool
def search_hotels(city:str)->str:
    """Search Hotels - returns long responses to use more tokens"""
    
    return f"""Hotels in {city}": 
            1. Grand Hotel 
            2. City In
            3. Budget Stay
            """

In [26]:
agent_token_count = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(model="gpt-4o-mini",
                                trigger=("tokens",300),
                                keep=("tokens",100))
    ]
)

In [20]:
def token_count(messages):
    tot = sum(len(str(m.content)) for m in messages)
    return tot//4

In [25]:
cities = ["NYC", "Delhi","Pune", "Chennai", "Dubai"]

for city in cities:
    response = agent_token_count.invoke(
        {"messages": [HumanMessage(content=f"Find hotel in {city}")]},
        config
    )
    token = token_count(response["messages"])
    
    print(f"{city}: - {token} tokens")

NYC: - 390 tokens
Delhi: - 459 tokens
Pune: - 412 tokens
Chennai: - 477 tokens
Dubai: - 426 tokens
